In [33]:
import os
import csv
import sys
import math
import scipy
import pickle
import librosa
import matplotlib
import numpy as np
import librosa.display
import IPython.display as ipd
matplotlib.use('TkAgg')
import matplotlib.pyplot as plt
import pandas as pd
import keras
from keras import layers
from keras import models
from keras.utils import to_categorical
from keras.models import Sequential
from keras.layers import Dense, Conv2D,Conv1D, Flatten,Dropout,MaxPooling2D
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
# from tensorflow.python.keras.layers.recurrent import LSTM
from keras.layers import LSTM
from keras.layers import Dense
from keras.optimizers import Adam
from sklearn.model_selection import KFold
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import GroupShuffleSplit

In [59]:
x_train = []
y_train = []
prog = []

nonprog = []

data = pd.read_csv('training_features.csv')
data.head()            
data = data.drop(['filename'],axis=1)
data = np.asarray(data)
print(data.shape)
y = data[:,0]
y[y=="prog"] = 0
y[y=="non_prog"] = 1
X = np.asarray(data)
X = X.astype('float32')
y = y.astype('int')

print(X.shape)
x_train, x_test, y_train, y_test = train_test_split( X, y, test_size=0.2, random_state=42)

X = np.reshape(X, (X.shape[0],1,X.shape[1]))
    

print("X_train shape ",x_train.shape)
print("X_test shape ",y_train.shape)

# y_train = to_categorical(y_train)
# y_test = to_categorical(y_test)
print("Y_train shape ",y_train.shape)
print(y_train)

# y = to_categorical(y)

(5113, 27)
(5113, 27)
X_train shape  (4090, 27)
X_test shape  (4090,)
Y_train shape  (4090,)
[1 1 1 ... 1 1 0]


In [50]:
#type of object in X_train in each column


<class 'float'>


In [68]:
input_shape = (1,X.shape[2])
kfold = KFold(n_splits=3, shuffle=True, random_state=45)
# cvscores = []
batch_size = 35
nb_epochs = 100
opt = Adam()

count = 1
for train, test in kfold.split(X, y):
  # Create model
    model = Sequential()
    model.add(LSTM(units=64, dropout=0.01, recurrent_dropout=0.35, return_sequences=True,input_shape=input_shape) )
    model.add(LSTM(units=32, dropout=0.01, recurrent_dropout=0.35, return_sequences=False))
    model.add(Dense(units=2, activation='sigmoid'))

    print("Compiling ...")
    model.compile(loss='categorical_crossentropy', optimizer=opt, metrics=['accuracy'])
    model.summary()

    print("Training ...")
    history = model.fit(X[train], to_categorical(y[train]), batch_size=batch_size, epochs=nb_epochs,validation_split=0.33)
    score, accuracy = model.evaluate(X[test], to_categorical(y[test]), batch_size=batch_size, verbose=1)
    print("Test loss:  ", score)
    print("Test accuracy:  ", accuracy)

    # Summarize history for accuracy
    plt.plot(history.history['accuracy'])  # Fix: Use 'accuracy' instead of 'acc'
    plt.plot(history.history['val_accuracy'])  # Fix: Use 'val_accuracy' instead of 'val_acc'
    plt.title('model accuracy')
    plt.ylabel('accuracy')
    plt.xlabel('epoch')
    plt.legend(['train', 'test'], loc='upper left')
    plt.show()
    plt.close()

    # Summarize history for loss
    plt.plot(history.history['loss'])
    plt.plot(history.history['val_loss'])
    plt.title('model loss')
    plt.ylabel('loss')
    plt.xlabel('epoch')
    plt.legend(['train', 'test'], loc='upper left')
    plt.show()
    plt.close()

    count += 1
model.save('Model.h5')

Compiling ...


Model: "sequential_14"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lstm_26 (LSTM)                  │ (None, 1, 64)          │        23,552 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_27 (LSTM)                  │ (None, 32)             │        12,416 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_13 (Dense)                │ (None, 2)              │            66 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 36,034 (140.76 KB)

 Trainable params: 36,034 (140.76 KB)

...

 Trainable params: 36,034 (140.76 KB)

 Non-trainable params: 0 (0.00 B)

 Optimizer params: 72,070 (281.53 KB)

Training ...
Epoch 1/100


ValueError: Unknown variable: <KerasVariable shape=(27, 256), dtype=float32, path=sequential_15/lstm_28/lstm_cell/kernel>. This optimizer can only be called for the variables it was originally built with. When working with a new set of variables, you should recreate a new optimizer instance.